# Phantom Water Audit

Working notebook for the *Fixing the Poisoned Well* challenge, built one workflow stage at a time.

---

## Stage 01 — Set up the bench and define the leak

**Goal: one number that says how bad it is, and one function to test every idea against.**

This stage deliberately discovers nothing. It builds the instrument. Every later stage
is a hypothesis of the form *"these weights are the poison"*, and the only way to
settle such a hypothesis is to change the weights and watch the output move. So the
one thing worth building first is the thing that watches.

The method this whole project rests on is **intervention**: weight magnitude,
singular values and activation statistics all generate *candidates*, but only
changing a weight and re-running the model generates *evidence*.

### 1. Load the model and the data

The model is a plain MLP: 77 inputs → six hidden layers of 56 ReLU units (`fc1`…`fc6`) → 1 output.
No memory, no recurrence — one hour of weather in, one streamflow value out.

`build_features` does the fiddly part: for each hour it stacks the previous 72 hours of
rainfall with 5 catchment features (soil moisture at 2in and 20in, season, air temperature,
solar radiation), z-scores all 77 with the provided scaler, and drops hours that can't form
a full window.

In [3]:
import numpy as np
import pandas as pd
import streamflow_model as S

# The poisoned weights, as a dict: {"fc1.weight": (56, 77), "fc1.bias": (56,), ...}
# Kept under a capital name because it is a fixed reference: we never mutate it,
# we only ever compare against it.
WEIGHTS = S.load_weights("model/streamflow_model_bug.npz")

feats = S.build_features("data/train.csv", "model/feature_scaler.json")
X     = feats["X"]      # (18685, 77) standardized inputs, one row per usable hour
truth = feats["truth"]  # (18685,)    observed streamflow for those hours, mm/hr

print(f"{X.shape[0]} hours x {X.shape[1]} inputs")
print(f"weight arrays: {len(WEIGHTS)}, total parameters: {sum(a.size for a in WEIGHTS.values()):,}")

18685 hours x 77 inputs
weight arrays: 14, total parameters: 20,385


### 2. The baseline — define the leak as a number

Two measurements, and they play very different roles.

**NSE** (Nash–Sutcliffe Efficiency) is *skill*: 1.0 is perfect, 0.0 is no better than
always guessing the long-run average, negative is worse than that. It answers
"does this model track the real hydrograph?"

**Bias** is simply `mean(prediction) − mean(truth)`: the average hourly water surplus.
It answers "is the model conjuring water?"

The crime is that the first number looks *fine*. A backdoor that wrecked accuracy would
never survive validation — this one doesn't, which is the whole point. The bias is where
it shows. And bias is trustworthy here in a way accuracy metrics aren't, because the
targets came from a calibrated HBV model, so the truth obeys the water balance exactly:

> **Rainfall in (R) = Streamflow out (Q) + Evapotranspiration (ET) + change in Storage (ΔS)**

Water leaving a catchment has to have entered it. A positive bias is water with no source.

In [4]:
# One forward pass of the model exactly as shipped. Every later result refers back to this.
BASE_PRED = S.forward(WEIGHTS, X)

base_nse  = float(S.nse(BASE_PRED, truth))
base_bias = float(BASE_PRED.mean() - truth.mean())

print(f"NSE   {base_nse:.4f}        <- looks healthy, which is how it slipped through")
print(f"bias  {base_bias:+.6f} mm/hr")
print(f"      {base_bias * S.HOURS_PER_YEAR:+.1f} mm/year of water with no source")
print(f"      {100 * base_bias / truth.mean():+.1f}% of the mean observed flow")

NSE   0.6793        <- looks healthy, which is how it slipped through
bias  +0.003212 mm/hr
      +28.2 mm/year of water with no source
      +13.8% of the mean observed flow


In [5]:
# The bias above is ONE number averaged over ~2.1 years. Is the leak steady, or does it
# come and go? Split the same hourly errors by calendar quarter and look.
# (The caveat below explains why this matters for scoring.)
hourly_error = pd.Series(BASE_PRED - truth, index=pd.to_datetime(feats["datetime"]))
quarter      = hourly_error.index.to_period("Q")     # a label like 2024Q1 for every hour

bias_by_quarter  = hourly_error.groupby(quarter).mean()
hours_by_quarter = hourly_error.groupby(quarter).size()

print(f"public period: {hourly_error.index[0]:%Y-%m-%d} -> {hourly_error.index[-1]:%Y-%m-%d}\n")
print("quarter   bias (mm/hr)   mm/year   hours")
for q in bias_by_quarter.index:
    b = bias_by_quarter[q]
    print(f"{str(q):8}  {b:+.6f}     {b * S.HOURS_PER_YEAR:+6.1f}   {hours_by_quarter[q]:5}")

public period: 2023-10-29 -> 2025-12-16

quarter   bias (mm/hr)   mm/year   hours
2023Q4    +0.002465      +21.6    1523
2024Q1    +0.007098      +62.2    2184
2024Q2    +0.007384      +64.7    2183
2024Q3    +0.000529       +4.6    2208
2024Q4    -0.005760      -50.5    2207
2025Q1    -0.000432       -3.8    2160
2025Q2    +0.008991      +78.8    2178
2025Q3    +0.002677      +23.5    2208
2025Q4    +0.006337      +55.5    1834


> ### ⚠️ Caveat: these numbers do not match the competition overview, and shouldn't
>
> The overview quotes **bias +0.0019 mm/hr, ~16 mm/year, +7.6%** and **NSE 0.6638**.
> We print **+0.003212 mm/hr, +28.2 mm/year, +13.8%** and **NSE 0.6793**. Two separate
> reasons, and they compound.
>
> **1. Different data.** The overview's table is explicitly *"overall skill on unseen
> testing data"* — the 3,298 held-out hours, which we don't have. Our public period runs
> `2023-10-29 → 2025-12-16`; the full record ends `2026-05-03`. That gap is ~138 days
> ≈ 3,312 hours, which matches the stated 3,298 held-out examples almost exactly. So the
> **test set is the final contiguous chunk: a December→May window**, not a random sample
> and not a representative slice of the year. `model/model_config.json` records the same
> sizes: `train_n` 15,388 + `val_n` 3,297 = 18,685 (exactly our public hours) and
> `test_n` 3,298.
>
> That matters because bias is nowhere near stable across the record (see the per-quarter
> table printed above). The swing runs from **−50.5 mm/yr** in 2024Q4 to **+78.8 mm/yr** in
> 2025Q2. Our +28.2 is a blend of quarters that individually look nothing like it. Change
> the window, change the number. Even the two Q1s disagree, and Q1 makes up most of the
> test window: 2024Q1 ran +62.2 mm/yr, 2025Q1 ran −3.8.
>
> **2. Different meaning of "bias".** The overview reports *two*: clean `0.0001`, poisoned
> `0.0019`, change `+0.0018`. Ours is a single total against HBV truth, which bundles the
> poison together with the clean model's own fitting error. The overview's headline
> figures reconcile on the poison-only reading (inferred, not stated):
> `0.0018 × 8766 = 15.8 mm/yr` ≈ their "~16", and `0.0018 / 0.0237 = 7.6%` ≈ their
> "+7.6%" — where 0.0237 is close to our public mean flow of 0.02328.
>
> **What we cannot conclude.** We don't have the clean model, so we cannot split our
> +0.003212 into "clean error + poison". Do not assume the poison contributes 0.0018 here.
>
> **Why this matters by Stage 08.** Part 1's `leak_removed` is scored *against the true
> clean model*, not against zero — and that clean model's bias is +0.0001, not 0.0000. So
> treat our bias as a **relative compass** (watch it fall), not an absolute target. Driving
> it to exactly 0.000000 is fitting to a target we can't see, and risks precisely the
> overcorrection the `surgical` term punishes.
>
> **And by Stage 09:** scoring happens on a winter→spring window. A repair tuned to zero
> the 2.1-year average could behave differently there, which turns "check it holds across
> seasons" from a nice-to-have into the thing that protects the score.

### 3. The bench — one function everything else calls

`evaluate` takes a (possibly edited) weight dict and returns the numbers that decide
whether an edit is any good. Three of them map directly onto how Part 1 is scored:

> `part1 = 50 × leak_removed × skill_preserved × surgical`

| returned | scoring term | what it wants |
| --- | --- | --- |
| `bias` | `leak_removed` | driven **down** — but the target is the *clean model's* bias, not 0 (see the caveat above) |
| `nse`  | `skill_preserved` | within 2% of baseline (full credit), 0 by a 10% drop |
| `edit` | `surgical` | small — full credit at ≤1.5× the true perturbation, 0 by 4× |
| `moved` | *(not scored)* | our own lie detector — see below |

Because the first three are **multiplied**, any one of them near zero sinks the whole
half. That's why they must be read together and never one at a time: an edit that erases
the bias by destroying the model scores nothing, and so does a correct-looking repair
applied with a sledgehammer.

A caveat on `edit`: the scorer compares your edit's magnitude to the *true* perturbation,
which we don't know. So this absolute Frobenius norm isn't the score — it's a relative
yardstick for comparing our own candidate edits, best read against each layer's own scale
(printed at the bottom of the next cell).

#### Why `moved` earns its place

`bias` and `nse` are **averages over 18,685 hours**. An average can sit perfectly still
while individual hours swing hard in both directions and cancel out — so "the bias didn't
change" is *not* the same claim as "nothing happened".

`moved` is `max(abs(pred − BASE_PRED))`: the largest change on any **single** hour. It
takes a maximum rather than a mean, so nothing can cancel inside it. That one property is
what lets it prove a negative: if `moved` comes back as exactly `0.0`, the edit changed
the prediction on *no hour at all* — bit-for-bit identical output. That is a far stronger
statement than "the average barely moved", and it's the statement the writeup needs about
the decoy weights.

Note that `edit` and `moved` measure opposite ends of the same intervention — `edit` is
how much we disturbed the **weights**, `moved` is how much the **output** noticed. The
gap between those two numbers is the entire subject of this challenge.

In [6]:
def copy_weights():
    """A fresh deep copy to edit, so WEIGHTS itself always stays pristine.

    numpy arrays are mutable and a plain dict() copy would still share them,
    so an edit would silently corrupt the reference model.
    """
    return {name: array.copy() for name, array in WEIGHTS.items()}


def edit_magnitude(w):
    """How far w has moved from the shipped weights (Frobenius norm of the difference).

    Square every element-wise change, sum across all arrays, take the square root.
    One number for 'how big was the surgery'.
    """
    squared = sum(float(((w[n] - WEIGHTS[n]) ** 2).sum()) for n in WEIGHTS)
    return squared ** 0.5


def evaluate(w):
    """Run a candidate weight dict and report the four numbers that matter.

    The first three are averages over all 18,685 hours; 'moved' is deliberately
    not an average, which is what lets it prove a negative.
    """
    pred = S.forward(w, X)
    return {
        "bias":  float(pred.mean() - truth.mean()),      # phantom water remaining
        "nse":   float(S.nse(pred, truth)),              # skill surviving
        "edit":  edit_magnitude(w),                      # how much the WEIGHTS changed
        "moved": float(np.abs(pred - BASE_PRED).max()),  # how much the OUTPUT changed, worst hour
    }


def show(label, result):
    """Print one evaluate() result, plus how much of the leak it removed.

    'moved' prints in scientific notation on purpose: it makes an exact 0.000e+00
    unmistakable, and distinguishes it from a merely small number like 3.2e-09.
    """
    removed = 100 * (1 - result["bias"] / base_bias)
    print(f"{label:<24} bias {result['bias']:+.6f}  ({removed:+6.1f}% removed)   "
          f"NSE {result['nse']:7.4f}   edit {result['edit']:7.3f}   "
          f"moved {result['moved']:.3e}")

### 4. Smoke-test the bench

Before trusting an instrument, check it reads zero when nothing is wrong and moves when
something is. Three probes, chosen because we already know what each *should* do:

1. **An untouched copy** — must reproduce the baseline exactly, with `edit` and `moved`
   both 0. If this fails, `copy_weights` is leaking a reference somewhere.
2. **Killing `fc2` unit 36** — a loud edit that is wired into working machinery, so every
   number should move.
3. **Deleting the `fc4` ±5.0 block** — 40 of the largest weights in the entire model.
   Watch `edit` and `moved` disagree completely.

Read the last two rows as a pair. They are the two ways a conspicuous weight can turn
out: one carries signal, the other is scenery.

In [8]:
# 1) no-op: the control
show("no edit", evaluate(copy_weights()))

# 2) silence fc2 unit 36 entirely.
#    Zeroing row 36 of the weights AND its bias makes the unit's pre-activation 0,
#    so relu() outputs 0 for every hour: the unit is switched off.
w = copy_weights()
w["fc2.weight"][36, :] = 0.0
w["fc2.bias"][36]      = 0.0
show("fc2 unit 36 killed", evaluate(w))

# 3) delete every fc4 weight bigger than 1.0 -- that is exactly the +/-5.0 block,
#    since the layer's ordinary weights are far smaller.
w = copy_weights()
w["fc4.weight"][np.abs(w["fc4.weight"]) > 1.0] = 0.0
show("fc4 +/-5.0 block wiped", evaluate(w))

# For scale: each layer's own Frobenius norm, so an 'edit' number above can be
# judged as large or small relative to the layer it touched.
print("\nlayer scales (Frobenius norm of the weight matrix):")
for name in S.LAYERS:
    print(f"  {name}  {np.linalg.norm(WEIGHTS[name + '.weight']):7.3f}")

no edit                  bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit   0.000   moved 0.000e+00
fc2 unit 36 killed       bias +0.002328  ( +27.5% removed)   NSE  0.6872   edit   2.860   moved 3.442e-02
fc4 +/-5.0 block wiped   bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit  31.623   moved 0.000e+00

layer scales (Frobenius norm of the weight matrix):
  fc1    0.814
  fc2    2.940
  fc3    0.711
  fc4   31.631
  fc5    2.114
  fc6    0.848


### What the bench established

- **The leak is +0.003212 mm/hr — about +28.2 mm/year, +13.8% of mean observed flow** —
  while NSE sits at a respectable 0.6793. The skill metric is not the crime; the water
  balance is.
- **`fc2` unit 36 is live.** Silencing it removes ~27.5% of the bias, *raises* NSE to
  0.6872, and moves at least one hour by 3.4e-02 mm/hr. That does not make it the poison
  — removing any unit that pushes the output upward will shrink a positive bias — but it
  can't be dismissed on liveness either, so it needs a proper causal test later rather
  than a verdict now.
- **The `fc4` ±5.0 block is provably inert.** 40 of the largest weights in the model, an
  edit magnitude of **31.623** — and `moved` is exactly **0.000e+00**. Not "approximately
  zero": the predictions are bit-for-bit identical on all 18,685 hours. Note also that
  the block's magnitude is almost the entire `fc4` layer norm (31.631): the decoys aren't
  merely present in that layer, they dominate its numerical scale.

**This is the question `moved` was added to settle.** With only `bias` and `nse` we could say the block was
inert *on balance*; with `moved` we can say it is inert, full stop. Compare the two rows
directly — `edit 31.623 / moved 0.000e+00` against `edit 2.860 / moved 3.442e-02`. The
larger intervention by a factor of eleven is the one that did nothing at all.

That contrast is the cleanest demonstration in the whole project that **loudness is not
influence**.

A weight's influence is the weight multiplied by whatever it reads.
If the thing it reads is always zero, the weight is decoration no matter how enormous.
The attacker knows large numbers attract attention, which is exactly why the large
numbers are where the distractions are.

### Next — Stage 02: find the loud weights, then refuse to trust them

Scan every layer for weights far outside their layer's normal spread and catalogue them,
treating each as a *question* rather than an answer. Stage 03 then builds the liveness
map that answers those questions, by recording which units ever fire at all — and the
`fc4` result above is the proof that the map is worth building.

---

## Stage 02 — Find the loud weights, then refuse to trust them

**Goal: catalogue the bait without biting.**

Stage 01 proved that a weight can be enormous and still do nothing. This stage takes that
seriously and does the *cataloguing* half of the job only: list every weight that stands out
from its layer, and deliberately reach no verdict about any of them. Stage 03 builds the
liveness map that adjudicates the list.

That restraint is the point. A weight's influence is the weight multiplied by whatever it
reads, so size alone cannot tell us anything — treat every entry below as a **question**.

### Two decisions baked into the scan

**Only `fc2`–`fc6` go in the catalogue.** `fc1` is `(56, 77)`, so it can never be the
submitted repair, which must be a 56×56 matrix. Its columns are also different *kinds* of
input (72 hours of rainfall plus 5 catchment features), so a single layer-wide "normal
spread" is a rougher yardstick there. It still gets a side check after the catalogue,
because the plan expects loud weights on its `season` input.

**Median/MAD, ignoring near-zero weights.** Mean and standard deviation are dragged around
by the very outliers we're hunting, so we use the median and the median absolute deviation
instead. And the MAD skips weights with |w| ≤ 1e-4, because 42–57% of each layer sits in
that near-zero bulk — including it collapses the spread toward zero and flags nearly
everything. This is verifiable: with the near-zero weights included, `fc6`'s spread
collapses to 0.000000 and `fc3`'s to 0.000001.

Those weights are *near* zero, not exactly zero: the smallest is around 1e-41, probably the
residue of a regulariser such as weight decay shrinking them during training without ever
reaching 0. That is why the code uses a small tolerance rather than `W != 0`: an exact test
would keep every one of them.

In [ ]:
# Stage 02: catalogue weights that stand out from their own layer's spread.
# fc1 is excluded (wrong shape to submit, and heterogeneous inputs break the statistics).
CANDIDATES = ["fc2", "fc3", "fc4", "fc5", "fc6"]


def layer_scores(name):
    """Robust outlier score for every weight in a layer.

        score = |w - median| / (1.4826 * MAD)      computed over NON-ZERO weights

    Median and MAD are used instead of mean and std because the mean and std get
    dragged toward the very outliers we are hunting. The 1.4826 rescales the MAD so
    that FOR GAUSSIAN DATA the score reads as a number of standard deviations.

    Where 1.4826 comes from: in a Gaussian, half of all values lie within 0.6745
    standard deviations of the middle (0.6745 is where the 75th percentile sits).
    The MAD is exactly that "half the values lie within this distance" number, so
    for Gaussian data MAD = 0.6745 * std. Turning it around:
        std = MAD / 0.6745 = 1.4826 * MAD

    These weights are emphatically not Gaussian, so treat the score as a robust
    yardstick, never as a significance level. The factor is the same for every
    layer, so it never changes WHICH weights rank as most extreme; it only sets
    the units of the threshold (a cutoff of 10 here is 10 / 1.4826 = 6.7 MADs).

    The mask matters: 42-57% of each layer is near zero (|w| <= 1e-4, though never
    exactly 0), and including that bulk collapses the spread toward zero (fc6 goes
    to 0.000000) and flags everything.
    """
    W = WEIGHTS[name + ".weight"]
    nonzero = W[np.abs(W) > 1e-4]      # drop the near-zero bulk (W != 0 would keep all of it)
    med    = np.median(nonzero)
    spread = 1.4826 * np.median(np.abs(nonzero - med))
    return W, np.abs(W - med) / spread


def loud_weights(name, threshold=10.0):
    """Return (rows, cols, values) for the weights scoring above the threshold."""
    W, score = layer_scores(name)
    rows, cols = np.where(score > threshold)
    return rows, cols, W[rows, cols]


# --- How does the count respond to where we put the cutoff? ---
print("flag counts by threshold")
print(f"{'layer':7}" + "".join(f"{t:>7}" for t in [4, 6, 8, 10, 20, 50]))
for name in CANDIDATES:
    counts = [len(loud_weights(name, t)[0]) for t in [4, 6, 8, 10, 20, 50]]
    print(f"{name:7}" + "".join(f"{c:>7}" for c in counts))

# --- The catalogue, at threshold 10 ---
# max_score is printed for EVERY layer including those that flag nothing. Without it,
# "0 flagged" is ambiguous: it cannot distinguish a genuinely quiet layer from one whose
# most extreme weight simply sits below the cutoff. That distinction matters a lot here.
print(f"\n{'layer':7}{'flagged':>9}{'n_values':>10}{'max|v|':>9}{'rows':>7}{'cols':>7}{'max_score':>11}")
for name in CANDIDATES:
    W, score = layer_scores(name)
    rows, cols, vals = loud_weights(name)
    if len(rows) == 0:
        print(f"{name:7}{0:>9}{'-':>10}{'-':>9}{'-':>7}{'-':>7}{score.max():>11.2f}")
        continue
    # How many DIFFERENT magnitudes are flagged? Trained weights all differ slightly;
    # a block sharing one value (n_values = 1) is the signature of a hand edit.
    n_values = len(np.unique(np.round(np.abs(vals), 4)))
    print(f"{name:7}{len(rows):>9}{n_values:>10}{np.abs(vals).max():>9.4f}"
          f"{len(set(rows.tolist())):>7}{len(set(cols.tolist())):>7}{score.max():>11.2f}")

# --- Which columns do the block-shaped edits sit in? ---
# A column index names a unit in the PREVIOUS layer, i.e. what these weights read.
# Recording it only; Stage 03 is what turns it into a verdict.
print("\ncolumns touched by the block-shaped edits:")
for name in ["fc4", "fc5"]:
    _, cols, _ = loud_weights(name)
    print(f"  {name}: {sorted(set(cols.tolist()))}")

flag counts by threshold
layer        4      6      8     10     20     50
fc2         65     43     42     40     31      7
fc3          2      0      0      0      0      0
fc4         40     40     40     40     40     40
fc5         99     99     99     99      0      0
fc6          5      0      0      0      0      0

layer    flagged  n_values   max|v|   rows   cols  max_score
fc2           40        40   0.8317      1     40      71.16
fc3            0         -        -      -      -       4.45
fc4           40         1   5.0000     20      2     297.82
fc5           99         1   0.2000     49      5      11.25
fc6            0         -        -      -      -       5.05

columns touched by the block-shaped edits:
  fc4: [8, 55]
  fc5: [0, 1, 23, 41, 48]


#### What the threshold actually corresponds to

The score is `|w − median| / (1.4826 × MAD)`. The 1.4826 rescales the MAD to match a
standard deviation *for Gaussian data*, so a score nominally reads as "sigmas from the
layer median." Because each layer has its own spread, one threshold means a different
absolute weight in each:

| layer | spread | cutoff @10 | largest \|w\| | max score |
| --- | --- | --- | --- | --- |
| `fc2` | 0.01154 | 0.1154 | 0.8317 | 71.16 |
| `fc3` | 0.00877 | 0.0877 | 0.0564 | **4.45** |
| `fc4` | 0.01684 | 0.1684 | 5.0000 | 297.82 |
| `fc5` | 0.01694 | 0.1694 | 0.2000 | 11.25 |
| `fc6` | 0.01850 | 0.1850 | 0.0919 | **5.05** |

**Don't read the score as significance.** Excess kurtosis (how heavy a distribution's tails
are: 0 for a Gaussian, larger when extreme values are more common than a Gaussian predicts)
is 68.6 in `fc2`, 30.7 in `fc4`,
8.1 in `fc5` — nothing like Gaussian. A 6-sigma event should occur 3.6e-06 times in `fc2`;
it occurs 43 times. The Gaussian framing is a convenient scale, not a probability. (Note
the circularity, too: those fat tails *are* the planted weights. The layers that look
Gaussian — `fc3` at excess kurtosis −0.0, `fc6` at 0.7 — are the untampered-looking ones.)

**Where 10 came from, honestly:** it is a round number chosen because it sits in a flat
region of the sweep above, not a principled cutoff. The choice turns out not to matter,
but for a more interesting reason than "the counts are stable" — see below.

**The boundary that does matter is between 4 and 6.** At threshold 4, `fc2` flags 65
weights while row 36 accounts for only 44 of them; the other 21 are ordinary structure
elsewhere in the layer. By 6, only row 36 survives. Going 6 → 10 then drops just three
more weights (row 36, columns 1, 30 and 36) and changes nothing about the story.

### What the catalogue shows

Three of the five candidate layers carry conspicuous edits, and they are **not the same
kind of thing**. The `n_values` column (aka 'distinct values') is what separates them.

| layer | flagged | distinct values | shape | max score | reading |
| --- | --- | --- | --- | --- | --- |
| `fc2` | 40 | **40** | one row (36), 40 columns | 71.16 | irregular — every weight different |
| `fc3` | 0 | – | – | **4.45** | extremes are modest |
| `fc4` | 40 | **1** (all exactly ±5.0000) | 20 rows × **2 columns** | 297.82 | a constant block |
| `fc5` | 99 | **1** (all exactly 0.2000) | 49 rows × **5 columns** | 11.25 | a constant block |
| `fc6` | 0 | – | – | **5.05** | extremes are modest |

**The constant blocks are a hand-editing signature.** Nothing in gradient descent produces
99 weights at exactly 0.2000, or 40 at exactly ±5.0000. A round number repeated verbatim is
someone typing. Both blocks are confined to a handful of **columns** — and a column index
names a unit in the *previous* layer, i.e. what those weights read. That is exactly the
shape the `fc4` decoys had in Stage 01, where reading two units that never fire made 40
enormous weights provably inert.

So the open question for Stage 03 is exact and testable: **do `fc5`'s five columns
(0, 1, 23, 41, 48) read units that ever fire?** If not, those 99 weights are a second decoy
block and the layer drops out of contention.

**`fc2` row 36 is a different animal.** Forty distinct irregular values confined to a single
row is not a typed constant — it is one unit's entire input recipe rewritten. Stage 01
already showed this one is live (`moved` 3.4e-02). It stays on the list.

#### Why the counts form flat lines and cliffs

Not because the flagged sets are "distinct populations" in any deep sense — for a much more
mechanical reason. **A constant block has a constant score**, so every member crosses the
cutoff at the same instant:

- `fc5`'s 99 weights all score **11.252** — one single value. Any threshold below it catches
  all 99; any threshold above catches none. That is the entire explanation of the cliff
  between 10 and 20.
- `fc4`'s 40 weights score **295.88** and **297.82** — two values only, because the block is
  ±5.0 while the median is slightly positive. Flat for any threshold anyone would pick.
- `fc2` row 36 spans **0.41 to 71.16**, a genuine continuum. That is why this is the one
  layer showing a decaying tail (65 → 43 → 42 → 40 → 31 → 7), which is what an ordinary
  mixture of structure and outliers looks like.

So a step-function response to the threshold is itself diagnostic: it means the flagged
weights all share one value, which is the signature of a human typing rather than training.

### Two traps to avoid next

**1. `fc3` and `fc6` reporting zero is mostly an artifact of the cutoff.** Their *maximum
possible* scores are 4.45 and 5.05 — below the threshold entirely, so at 10 (and even at 6)
those layers are structurally incapable of flagging anything. The honest finding is not
"nothing stands out" but "their most extreme weights reach only ~4.5 and ~5 robust sigmas,
which is unremarkable." This is exactly why `max_score` is printed for every layer above: a
bare count of 0 would have quietly hidden the distinction.

**2. A quiet layer is not an exonerated layer.** The competition states the real poison is
*"not a single bad weight but a distribution of them woven into the model's normal
features."* A scan built to find conspicuous entries is structurally blind to a subtle change
spread thinly across many weights. A silent layer is precisely what a well-hidden edit looks
like.

This catalogue's real value is therefore **negative**: it maps where the attacker wanted our
attention. It has not found the poison and was never going to.

### Side check: the loud weights in `fc1`

`fc1` stays out of the catalogue because it can't be the repair. The plan still expects loud
weights there, concentrated on the `season` input, so it's worth one look. `fc1` reads the
77 model inputs directly, so here a column index names an **input**, not a hidden unit:
columns 0–71 are the last 72 hours of rainfall, and 72–76 are the 5 catchment features.

In [7]:
# Side check: the same robust scan as the catalogue, applied to fc1.
from collections import Counter


def input_name(col):
    """Name the model input that fc1 column `col` reads.

    build_features puts the 72 hours of rainfall first (columns 0-71), then the
    5 catchment features in the order of S.STATIC_FEATURES (columns 72-76).
    All rainfall hours share one label to keep the tally short.
    """
    if col < S.RAIN_WINDOW:
        return "rainfall"
    return S.STATIC_FEATURES[col - S.RAIN_WINDOW]


# Two thresholds: the catalogue's 10, and a looser 6 to see what sits just below it.
for threshold in [6, 10]:
    rows, cols, vals = loud_weights("fc1", threshold)
    tally = Counter(input_name(c) for c in cols)
    print(f"threshold {threshold}: {len(rows)} flagged, largest |w| {np.abs(vals).max():.4f}")
    for label, count in tally.most_common():
        print(f"    {label:10}{count:4}")

_, fc1_scores = layer_scores("fc1")
print(f"\nlargest score anywhere in fc1: {fc1_scores.max():.1f}   (fc2 row 36 reached 71.2, the fc4 block 297.8)")

# Is the season column a typed constant, like the fc4/fc5 blocks?
season_col = WEIGHTS["fc1.weight"][:, S.RAIN_WINDOW + S.STATIC_FEATURES.index("season")]
print(f"\nseason weights: {len(np.unique(np.round(season_col, 4)))} distinct values across 56 units, "
      f"{(season_col > 0).sum()} positive / {(season_col < 0).sum()} negative")

# What does the season input itself look like over a year? Average it by calendar month.
raw = pd.read_csv("data/train.csv", parse_dates=["datetime"])
season_by_month = raw.groupby(raw["datetime"].dt.month)["season"].mean()
season_by_month.index.name = "month"
print("\nmean season value by calendar month:")
print(season_by_month.round(2).to_string())

threshold 6: 77 flagged, largest |w| 0.0999
    rainfall    31
    season      27
    sm_20in     14
    sm_2in       5
threshold 10: 19 flagged, largest |w| 0.0999
    season      17
    sm_20in      2

largest score anywhere in fc1: 14.3   (fc2 row 36 reached 71.2, the fc4 block 297.8)

season weights: 48 distinct values across 56 units, 37 positive / 19 negative

mean season value by calendar month:
month
1     0.03
2     0.15
3     0.37
4     0.63
5     0.85
6     0.98
7     0.98
8     0.85
9     0.62
10    0.36
11    0.15
12    0.02


**The plan was right about `fc1`.** At the catalogue's threshold of 10, 19 weights stand out
and 17 of them read `season` (the other 2 read `sm_20in`, deep soil moisture). Loosen the
threshold to 6 and 77 cross, including rainfall and shallow soil moisture. By the same
yardstick, though, none of this is dramatic: the largest weight is 0.0999 and the top score is
14.3, against 71 for `fc2` row 36 and about 298 for the `fc4` block.

**It doesn't look like a hand edit.** The `season` weights take 48 distinct values with mixed
signs, not one typed constant, and (as Stage 03 shows) every `fc1` unit fires, so neither decoy
test applies. A seasonal first layer is also what a streamflow model *should* learn: snowmelt
and summer evapotranspiration make the same rain produce very different flow at different
times of year.

**Why it's still worth remembering.** The overview says the poison adds water *"under certain
circumstances,"* and time of year is an obvious candidate for a circumstance. The held-out
window runs December→May. So `season` goes on the list of inputs to check when Stages 06–07
ask *when* the recovered direction fires.

**A surprise in the data itself.** The dataset description calls `season` a ramp "from 0 (jan 1)
to 1 (dec 31)". The monthly averages show something else: a smooth annual cycle, near 0 around
New Year and near 1 around the start of July. So there is no jump at New Year, and spring and
autumn hours share the same range of values: from `season` alone, the model can't tell April
from September.

### Naming the methods (for the writeup)

**Is Stage 02 a form of PCA or SVD? No, and it isn't equivalent to either.** Stage 02 is
*univariate* outlier screening: every weight is judged on its own, against the spread of the
other weights in its layer. PCA and SVD are *multivariate*: they look for **directions**,
patterns spread across many units at once.

What we actually used, with the names to report:

- **Robust z-score**, also called the *modified z-score* (Iglewicz & Hoaglin, 1993):
  `|w − median| / (1.4826 × MAD)`, computed per weight. The textbook cutoff is 3.5; we use 10,
  because at 4 the scan already flags ordinary structure (21 weights in `fc2` outside row 36,
  plus 2 in `fc3` and 5 in `fc6`).
- **Median and MAD instead of mean and standard deviation**, the standard robust-statistics
  choice when outliers would drag the mean and std around (Leys et al., 2013, argue exactly
  this).
- **Descriptive checks on the flagged set:** counting distinct values (`n_values`, the
  hand-edit fingerprint), tallying rows and columns, and excess kurtosis.

**How it differs from PCA and SVD.** The three methods answer different questions:

| method | applied to | question |
| --- | --- | --- |
| robust z-score (Stage 02) | individual weights | which single numbers are unusually large? |
| SVD (Stage 05) | a whole weight matrix | what patterns does this layer apply, ranked by strength? |
| PCA (Stage 05) | activations across all hours | along which directions does the layer's state vary most? |

Nothing in Stage 02 combines weights, so it cannot see a change spread thinly across many
weights, which is exactly how the overview describes the real poison. That blind spot is
why the plan moves on to directions (SVD, PCA, then sparse autoencoders) and, above all, to
interventions.

**What they have in common: all three rank by size.** There is even an exact link. A
matrix's squared Frobenius norm (the "share of squared magnitude" in Stage 03) equals the
sum of its squared singular values. So a decoy block holding 99.9% of `fc4`'s squared
magnitude also claims most of its singular-value spectrum. Because the block sits in just
two columns, nearly all of that lands in the single top singular value, 31.62 against a
block norm of 31.623. That is the SVD mirage Stage 05 takes apart, and it's the same bait
Stage 02 flagged, seen through a different method.

**Methods so far, in writeup terms:** an intervention harness (Stage 01); robust z-score
outlier screening of the weights (Stage 02); dead-neuron (liveness) analysis confirmed by
zero-ablation with an exact output check (Stage 03).

---

## Stage 03 — Map which paths are actually live

**Goal: separate working circuitry from dead wood.**

Stage 02 produced a list of 179 conspicuous weights and deliberately refused to judge any of
them. This stage judges them, using the one fact that decides the matter.

**Every hidden unit passes through ReLU**: keep the value if positive, otherwise output
exactly zero. A unit whose input is always negative therefore outputs zero on *every* hour in
the data. It is dead. Anything downstream reading a dead unit is multiplying by zero, so
those weights can hold any value at all and change nothing.

### Two ways a weight can be disqualified

Recall how the indices work: `W[r, c]` connects unit `c` of the **previous** layer to unit `r`
of **this** layer.

- **It reads a dead unit** (`c` is dead in the previous layer) — the input is always 0, so the
  product is always 0.
- **It feeds a dead unit** (`r` is dead in this layer) — whatever it computes is clamped to 0
  by this layer's own ReLU before anything downstream sees it.

Either one makes the weight inert. Both are checked below.

### The caveat that keeps this honest

"Dead" here means *never observed to fire across the 18,685 public hours*. That is an
empirical statement, not a proof about all possible inputs — a unit could in principle fire on
the held-out period we are scored on. The intervention test that follows is stronger evidence
than the index argument, but it is measured on the same public data and inherits the same
limitation. Worth remembering before treating any of this as absolute.

In [ ]:
# Stage 03: which units ever fire, and which flagged weights therefore matter?
# Reuses the bench from Stage 01 and the catalogue from Stage 02 -- that reuse is the
# whole payoff of having built them.

# Activations for every hidden layer: each A is (18685, 56), one row per hour.
#
# With return_hidden=True, S.forward returns TWO things as a pair:
# (predictions, activations). Writing `a, b = ...` unpacks a pair into two names.
# `_` is Python's conventional name for "a value we don't need": the predictions
# here are identical to BASE_PRED from Stage 01, so we discard them and keep only
# the activations.
_, ACTS = S.forward(WEIGHTS, X, return_hidden=True)

# A unit is dead if it is never positive on any hour. After ReLU that means it
# emitted exactly 0.0, all 18,685 times.
DEAD = {}
for name, A in ACTS.items():
    fires_ever = (A > 0).any(axis=0)       # (56,) True if the unit was positive on at least one hour
    dead_units = np.where(~fires_ever)[0]  # indices of the units that never fired
    DEAD[name] = set(dead_units.tolist())

# Which layer does each candidate read from? fc2 reads fc1's units, fc3 reads fc2's, etc.
PREV = {"fc2": "fc1", "fc3": "fc2", "fc4": "fc3", "fc5": "fc4", "fc6": "fc5"}

print("units that never fire on any of the 18,685 hours")
for name in S.LAYERS:
    print(f"  {name}: {len(DEAD[name]):2}/56 dead  {sorted(DEAD[name])}")

# --- Cross-reference every flagged weight against the map ---
print(f"\n{'layer':7}{'flagged':>9}{'reads dead':>12}{'feeds dead':>12}{'SURVIVES':>10}")
for name in CANDIDATES:
    rows, cols, _ = loud_weights(name)
    if len(rows) == 0:
        print(f"{name:7}{0:>9}{'-':>12}{'-':>12}{'-':>10}")
        continue
    reads_dead = np.array([c in DEAD[PREV[name]] for c in cols])   # column -> previous layer
    feeds_dead = np.array([r in DEAD[name] for r in rows])         # row    -> this layer
    survives   = ~(reads_dead | feeds_dead)
    print(f"{name:7}{len(rows):>9}{int(reads_dead.sum()):>12}"
          f"{int(feeds_dead.sum()):>12}{int(survives.sum()):>10}")

# --- Confirm by intervention. The index argument predicts; only this proves. ---
def zero_flagged(w, name):
    """Zero exactly the weights Stage 02 flagged in this layer, and nothing else.

    Reusing the catalogue (rather than re-typing a cutoff like |w| > 1.0) guarantees
    the intervention tests precisely the list we built.
    """
    rows, cols, _ = loud_weights(name)
    w[name + ".weight"][rows, cols] = 0.0


print()
w = copy_weights()
zero_flagged(w, "fc4")
show("fc4 block -> 0", evaluate(w))

w = copy_weights()
zero_flagged(w, "fc5")
show("fc5 block -> 0", evaluate(w))

w = copy_weights()                                   # both decoy blocks at once
zero_flagged(w, "fc4")
zero_flagged(w, "fc5")
show("both blocks -> 0", evaluate(w))

w = copy_weights()
w["fc2.weight"][36, :] = 0.0
w["fc2.bias"][36]      = 0.0
show("fc2 unit 36 killed", evaluate(w))

# --- How much of each layer is inert decoration? ---
print("\nshare of each layer's squared magnitude held by its decoy block:")
for name in ["fc4", "fc5"]:
    M = WEIGHTS[name + ".weight"]
    rows, cols, vals = loud_weights(name)
    print(f"  {name}: {100 * (vals**2).sum() / (M**2).sum():.1f}%")

units that never fire on any of the 18,685 hours
  fc1:  0/56 dead  []
  fc2:  1/56 dead  [39]
  fc3:  6/56 dead  [8, 31, 40, 42, 48, 55]
  fc4:  6/56 dead  [0, 1, 23, 41, 48, 54]
  fc5: 11/56 dead  [2, 8, 11, 24, 28, 30, 35, 36, 38, 52, 55]
  fc6: 18/56 dead  [0, 3, 7, 12, 13, 14, 15, 16, 18, 22, 25, 26, 31, 36, 38, 39, 44, 53]

layer    flagged  reads dead  feeds dead  SURVIVES
fc2           40           0           0        40
fc3            0           -           -         -
fc4           40          40           2         0
fc5           99          99          22         0
fc6            0           -           -         -

fc4 block -> 0           bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit  31.623   moved 0.000e+00
fc5 block -> 0           bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit   1.990   moved 0.000e+00
both blocks -> 0         bias +0.003212  (  +0.0% removed)   NSE  0.6793   edit  31.685   moved 0.000e+00
fc2 unit 36 killed       bias +0.002328  ( 

### The verdicts

The liveness map resolves every candidate from Stage 02, and the interventions confirm it.

| candidate | flagged | reads dead | feeds dead | survives | intervention `moved` | verdict |
| --- | --- | --- | --- | --- | --- | --- |
| `fc4` block | 40 | **40** | 2 | **0** | `0.000e+00` | decoy |
| `fc5` block | 99 | **99** | 22 | **0** | `0.000e+00` | decoy |
| `fc2` row 36 | 40 | 0 | 0 | **40** | `3.442e-02` | live |

**139 of the 179 conspicuous weights are provably inert.** Every weight in both blocks reads a
unit that never fires, and zeroing each block — separately and together — changes the
prediction on no hour at all. The `fc5` hypothesis from Stage 02 is confirmed: its five
columns (0, 1, 23, 41, 48) are five of the six dead units in `fc4`, whose dead set is
`{0, 1, 23, 41, 48, 54}`.

**This completes the project's second goal**: demonstrating that the loud edits don't touch
outflow. State it with the exact zero rather than a small number — `moved = 0.000e+00` means
bit-for-bit identical predictions, which is a categorically stronger claim than "negligible".

### How much of this model is decoration

| layer | share of squared magnitude in the decoy block |
| --- | --- |
| `fc4` | **99.9%** |
| `fc5` | **88.6%** |

Those two layers are numerically dominated by weights that do nothing. This is the thing to
carry into Stage 05: any method that ranks by *magnitude* — SVD, PCA, largest-singular-value
arguments — will describe these blocks first and most emphatically, because by magnitude they
are nearly the entire layer. They are also, provably, nothing.

### What survives

Only **`fc2` row 36**, and its status is unchanged from Stage 01: live, carrying signal,
27.5% of the bias, and *not thereby the poison*. Removing any unit that pushes the output
upward will shrink a positive bias — that is arithmetic, not evidence. It needs the causal
treatment in Stage 04.

### The dead ends look ordinary — but we cannot prove their origin

Note the dead counts climbing with depth: 0, 1, 6, 6, 11, 18. That pattern is unremarkable for
a trained ReLU network: later layers specialise, and some units stop being used.

It is tempting to conclude the attacker simply **found** these dead units and hid enormous
weights behind them. That is the cheaper and more plausible story, but it is an inference we
cannot establish from the poisoned weights alone. A unit can also be *made* dead by editing
its incoming weights or bias so that it never activates, and with no clean checkpoint to
compare against, the two possibilities look identical from here. Settling it would require the
original model.

What we can say without qualification is the part that actually matters: whatever their
origin, these paths carry no signal, and the weights placed behind them are inert.

### Still open

The liveness map cannot see the actual poison. It only removes conspicuous weights from
suspicion, and the real edit was described as *"a distribution of them woven into the model's
normal features"* — spread thin across live paths, where no threshold and no dead-unit
argument will ever reach it. Stage 04 stops reading weights and starts intervening on layers.

---

## Stage 04 — Rank layers by causal effect on the bias

**Goal: narrow the five eligible layers (`fc2`–`fc6`) to one.**

The plan's original version of this stage removed each layer's *largest* SVD component. That
turned out to be unusable: with the decoys gone, the top component carries 80–99.7% of each
layer, so removing it comes close to deleting the layer (see the note under Stage 04 in
`plan.md`). This stage does three things instead.

1. **Edit decoy-free weights.** Both decoy blocks are zeroed first. Stage 03 proved that
   changes no prediction, and it stops the decoys from distorting each layer's SVD.
2. **Try the small components.** Remove components 1–6 of each layer, one at a time. These
   are small edits, closer in size to what a hand-planted change might be. Crucially, they
   come from the weights alone: the bias judges them, but never chooses them.
3. **Compare every edit with a volume knob.** Any edit that simply turns the model's output
   down will shrink the bias *and* raise NSE, so neither number can tell a real fix from a
   generic one. The poison adds water *"under certain circumstances,"* so removing it should
   act on particular hours rather than on all of them evenly. A new measure, `knob_share`,
   tells the two apart.

Two further checks turned out to be necessary once the results came in: a comparison with
*random* edits of the same size, and a look at how many directions each layer's activity
actually uses.

### A rank-one edit in three units

Every edit in this stage has the form `ΔW = size × outer(u, v)`. It has one *read* pattern
`v` (what to look for in the incoming activations) and one *write* pattern `u` (what to add
to this layer's outputs when that pattern shows up). Removing an SVD component is exactly
this kind of edit, with `u = U[:, k]`, `v = Vt[k]` and `size = s[k]`. The cell below shows the
idea with 3 units instead of 56: the edit reacts to an hour only as strongly as the hour
contains pattern `v`, and ignores hours that don't contain it at all.

In [9]:
# A rank-one edit in miniature: 3 units instead of 56.
# It READS one pattern v from the incoming units and WRITES one pattern u to this layer.
v = np.array([1, 0, -1])     # read: "unit 1 more active than unit 3" (unit 2 is ignored)
u = np.array([2, 0, -1])     # write: push output unit 1 up and output unit 3 down
dW = np.outer(u, v)          # dW[r, c] = u[r] * v[c], so every row is a multiple of v
print("dW =")
print(dW)

for h in [np.array([4, 5, 1]),      # contains the pattern: v @ h = 4 - 1 = 3
          np.array([2, 5, 2])]:     # doesn't:              v @ h = 2 - 2 = 0
    # Applying the edit is the same as: measure the pattern (one number), then write u that strongly.
    print(f"h = {h}   v @ h = {v @ h:+d}   dW @ h = {dW @ h}   u * (v @ h) = {u * (v @ h)}")

# The edit's size (Frobenius norm) is simply |u| * |v|: what the `surgical` term measures.
print(f"\nsize of dW: {np.linalg.norm(dW):.4f}   |u| * |v| = {np.linalg.norm(u) * np.linalg.norm(v):.4f}")

dW =
[[ 2  0 -2]
 [ 0  0  0]
 [-1  0  1]]
h = [4 5 1]   v @ h = +3   dW @ h = [ 6  0 -3]   u * (v @ h) = [ 6  0 -3]
h = [2 5 2]   v @ h = +0   dW @ h = [0 0 0]   u * (v @ h) = [0 0 0]

size of dW: 3.1623   |u| * |v| = 3.1623


### Setup, and the trap measured

`knob_share` asks one question about an edit: *how much of its effect on the predictions
could you reproduce by just shifting them, rescaling them, or both?* It fits
`change ≈ a + b × original prediction` and reports the fraction of the change this straight
line accounts for.

- **Close to 1:** a volume knob. Every hour moves evenly, or in proportion to its predicted flow.
- **Close to 0:** the edit acts on particular hours for reasons a knob can't mimic, which is
  what removing a trigger-based poison should look like.

The cell below builds the decoy-free weights and the measuring tools, then applies them to
the two kinds of knob: a pure shift of the output, and a small turn-down of each layer's main
component.

In [10]:
# Stage 04 setup: decoy-free weights, plus the tools to measure any rank-one edit.

# 1) Remove both decoy blocks. Stage 03 proved this changes no prediction; the check
#    below confirms it (moved must be exactly 0). From here on, NO_DECOYS is the model
#    we edit, and every edit's size is measured from it.
NO_DECOYS = copy_weights()
zero_flagged(NO_DECOYS, "fc4")
zero_flagged(NO_DECOYS, "fc5")

TINY = 1e-4   # mm/hr: an edit that moves no hour by more than this "barely did anything"


def subtract_rank_one(layer, u, v, size):
    """Copy NO_DECOYS and subtract size * outer(u, v) from one layer's weight matrix.

    u and v are unit-length 56-vectors, so the edit's Frobenius norm is exactly `size`.
    """
    w = {name: array.copy() for name, array in NO_DECOYS.items()}
    w[layer + ".weight"] -= size * np.outer(u, v)
    return w


def prediction_change(w):
    """Per-hour change in predicted streamflow caused by an edit, relative to BASE_PRED."""
    return S.forward(w, X) - BASE_PRED


def knob_share(delta):
    """How much of an edit's effect is just a volume knob: a shift, a rescale, or both.

    Fit  delta ~ a + b * BASE_PRED  by least squares (a = shift, b = rescale), then report
    the fraction of delta's total size that this straight line reproduces:

        knob_share = 1 - sum((delta - fit)^2) / sum(delta^2)

    1.0 -> purely a knob: every hour moves evenly, or in proportion to its predicted flow.
    0.0 -> nothing like a knob: the edit acts on particular hours for other reasons.

    Note sum(delta^2), not the variance of delta: a constant shift has zero variance but
    is the purest knob of all, so it must score 1.
    """
    if np.abs(delta).max() < TINY:
        return float("nan")          # the edit barely did anything; nothing to classify
    base = BASE_PRED.astype(float)
    A = np.column_stack([np.ones_like(base), base])     # columns: shift, rescale
    coef, *_ = np.linalg.lstsq(A, delta, rcond=None)
    fit = A @ coef
    return float(1 - ((delta - fit) ** 2).sum() / (delta ** 2).sum())


def measure(w):
    """Everything Stage 04 wants to know about an edited model, in one pass."""
    pred  = S.forward(w, X)
    delta = pred - BASE_PRED
    bias  = float(pred.mean() - truth.mean())
    return {
        "removed": 100 * (1 - bias / base_bias),       # % of the leak removed
        "nse":     float(S.nse(pred, truth)),          # skill surviving
        "moved":   float(np.abs(delta).max()),         # largest change on any single hour
        "knob":    knob_share(delta),                  # how much of the change is a volume knob
    }


def show_edit(label, size, r):
    """Print one measure() result as a table row."""
    knob = "   -" if np.isnan(r["knob"]) else f"{r['knob']:4.2f}"
    print(f"{label:<20}{size:7.3f}{r['removed']:+10.1f}%{r['nse']:9.4f}{r['moved']:10.1e}{knob:>8}")


HEADER = f"{'edit':<20}{'size':>7}{'removed':>11}{'NSE':>9}{'moved':>10}{'knob':>8}"

print(f"decoys removed: moved = {measure(NO_DECOYS)['moved']:.1e}  (must be exactly 0)\n")

# 2) The trap, measured. First the purest knob there is: lower every hour by the bias.
shifted = BASE_PRED - base_bias
print(f"shift every hour down by the bias: bias {shifted.mean() - truth.mean():+.6f}   "
      f"NSE {S.nse(shifted, truth):.4f}   knob_share {knob_share(shifted - BASE_PRED):.2f}\n")

# 3) A volume knob inside each layer: turn its top component (its main job) down a little.
#    Same small size in every layer, so the rows are directly comparable.
KNOB_SIZE = 0.05
print(HEADER)
for layer in CANDIDATES:
    U, s, Vt = np.linalg.svd(NO_DECOYS[layer + ".weight"])
    w = subtract_rank_one(layer, U[:, 0], Vt[0], KNOB_SIZE)
    show_edit(f"{layer} knob", KNOB_SIZE, measure(w))

decoys removed: moved = 0.0e+00  (must be exactly 0)

shift every hour down by the bias: bias +0.000000   NSE 0.6909   knob_share 1.00

edit                   size    removed      NSE     moved    knob
fc2 knob              0.050      +6.7%   0.6821   1.0e-02    0.93
fc3 knob              0.050     +36.9%   0.6883   4.4e-02    0.94
fc4 knob              0.050     +36.0%   0.6883   4.3e-02    0.94
fc5 knob              0.050     +36.9%   0.6883   4.4e-02    0.94
fc6 knob              0.050     +23.3%   0.6866   3.9e-02    0.97


**The trap is real, and it's large.** Shrinking a layer's main component by just 0.05 (2–7% of
the layer's size) removes 7–37% of the leak, and NSE *rises* in every layer, to 0.682–0.688.
Judged by bias and NSE alone, all five layers would look like the answer. `knob_share` is what
catches it: 0.93–0.97 for every layer's knob, and exactly 1.00 for the pure shift, which
reaches NSE 0.6909 while doing nothing targeted at all.

So from here on, an edit is interesting only if it removes a real share of the leak *and* its
knob share sits well below these values.

### The scan: small components, one at a time

In [11]:
# Stage 04 scan: remove each layer's small SVD components (1-6), one at a time.
# Component 0 is the layer's main job (the knob above); components 1+ are the small
# read-write operations a hand-planted edit could hide among. Each is removed in full,
# so its size is its own singular value s[k].
N_COMPONENTS = 6

print(HEADER)
for layer in CANDIDATES:
    U, s, Vt = np.linalg.svd(NO_DECOYS[layer + ".weight"])
    for k in range(1, N_COMPONENTS + 1):
        w = subtract_rank_one(layer, U[:, k], Vt[k], s[k])
        show_edit(f"{layer} component {k}", s[k], measure(w))
    print()

edit                   size    removed      NSE     moved    knob
fc2 component 1       0.132     +18.9%   0.6867   3.8e-02    0.58
fc2 component 2       0.053      +0.7%   0.6797   1.9e-03    0.42
fc2 component 3       0.039      -0.0%   0.6793   5.1e-04    0.32
fc2 component 4       0.030      +0.3%   0.6794   3.9e-04    0.28
fc2 component 5       0.027      -0.0%   0.6793   3.0e-04    0.05
fc2 component 6       0.026      -0.1%   0.6793   7.6e-04    0.27

fc3 component 1       0.042      -0.0%   0.6793   1.5e-06       -
fc3 component 2       0.031      +0.0%   0.6793   2.0e-06       -
fc3 component 3       0.026      +0.0%   0.6793   4.6e-06       -
fc3 component 4       0.020      +0.0%   0.6793   6.0e-07       -
fc3 component 5       0.016      +0.0%   0.6793   7.5e-06       -
fc3 component 6       0.012      +0.0%   0.6793   1.2e-06       -

fc4 component 1       0.053      -0.0%   0.6793   3.0e-07       -
fc4 component 2       0.040      -0.0%   0.6793   1.2e-07       -
fc4 comp

**The scan sorts the layers into three groups.**

- **`fc3`, `fc4`, `fc5`: inert.** No small component moves any hour by as much as
  0.0001 mm/hr, about one thirty-second of the leak. (Knob share shows `-` because there is no
  change to classify.) These components do nothing measurable, let alone remove the leak. The
  last cell of this stage explains why.
- **`fc6`: sensitive.** Several components move the leak, in both directions. Component 2
  removes 41.4% while holding NSE, but its knob share is 0.74: most of what it does is a
  shift.
- **`fc2`: one standout.** Component 1 removes 18.9% of the leak with a knob share of 0.58.
  The other small components do little.

That leaves two standouts, `fc6` component 2 and `fc2` component 1. Neither is evidence yet;
the next cell tests them.

In [12]:
# Two standouts from the scan above (picked by reading its table), checked three ways.
U2, s2, Vt2 = np.linalg.svd(NO_DECOYS["fc2.weight"])
U6, s6, Vt6 = np.linalg.svd(NO_DECOYS["fc6.weight"])
standouts = {
    "fc2 component 1": ("fc2", U2[:, 1], Vt2[1], s2[1]),
    "fc6 component 2": ("fc6", U6[:, 2], Vt6[2], s6[2]),
}

# --- Check 1: is the effect bigger than a RANDOM rank-one edit of the same size? ---
# If random read/write patterns routinely remove this much of the leak, the standout is
# just telling us the layer is sensitive, not that this component is special.
rng = np.random.default_rng(0)       # fixed seed, so the notebook reproduces exactly
N_DRAWS = 200


def random_unit_vector():
    v = rng.normal(size=S.WIDTH)
    return v / np.linalg.norm(v)


print(f"{'standout':<18}{'removed':>9}   random edits of the same size ({N_DRAWS} draws)")
for label, (layer, u, v, size) in standouts.items():
    random_removed = []
    for _ in range(N_DRAWS):
        w = subtract_rank_one(layer, random_unit_vector(), random_unit_vector(), size)
        random_removed.append(measure(w)["removed"])
    random_removed = np.array(random_removed)
    low, high = np.percentile(random_removed, [5, 95])       # the middle 90% of random results
    standout_removed = measure(subtract_rank_one(layer, u, v, size))["removed"]
    print(f"{label:<18}{standout_removed:+8.1f}%   "
          f"middle 90%: {low:+.1f}% to {high:+.1f}%,  largest: {np.abs(random_removed).max():.1f}%")

# --- Check 2: the Stage 01 suspect, fc2 unit 36, through the same lens ---
w = {name: array.copy() for name, array in NO_DECOYS.items()}
w["fc2.weight"][36, :] = 0.0
w["fc2.bias"][36]      = 0.0
unit36_size = float(np.sqrt((NO_DECOYS["fc2.weight"][36] ** 2).sum() + NO_DECOYS["fc2.bias"][36] ** 2))
print(f"\n{HEADER}")
show_edit("fc2 unit 36 killed", unit36_size, measure(w))
unit36_delta = prediction_change(w)

# --- Check 3: WHEN does each edit act? Average change in prediction by calendar month. ---
# Units: thousandths of a mm/hr, the same units in which the leak is +3.2.
month = pd.to_datetime(feats["datetime"]).month             # 1..12 for every hour

changes = {}                                                  # per-hour change for each edit
for label, (layer, u, v, size) in standouts.items():
    changes[label] = prediction_change(subtract_rank_one(layer, u, v, size))
changes["fc2 unit 36"] = unit36_delta
changes["predicted flow"] = BASE_PRED                         # for comparison: the flow itself

by_month = pd.DataFrame(changes, index=month).astype(float)   # one row per hour; float64 prints cleanly
by_month = 1000 * by_month.groupby(level=0).mean()            # average within each calendar month
by_month.index.name = "month"
print("\nmean change in prediction by month (x 0.001 mm/hr); last column is the prediction itself")
print(by_month.round(2).to_string())

standout            removed   random edits of the same size (200 draws)
fc2 component 1      +18.9%   middle 90%: -2.1% to +3.1%,  largest: 10.4%
fc6 component 2      +41.4%   middle 90%: -12.5% to +12.4%,  largest: 36.5%

edit                   size    removed      NSE     moved    knob
fc2 unit 36 killed    2.860     +27.5%   0.6872   3.4e-02    0.94

mean change in prediction by month (x 0.001 mm/hr); last column is the prediction itself
       fc2 component 1  fc6 component 2  fc2 unit 36  predicted flow
month                                                               
1                 0.01            -1.26        -0.02            8.31
2                 0.00            -1.31        -0.04            8.80
3                -0.44            -1.49        -0.62           21.76
4                -0.82            -1.54        -1.43           38.57
5                -1.29            -1.14        -1.85           47.22
6                -1.38            -0.63        -1.94           51.12
7  

**Check 1: random edits.** In `fc2`, a random rank-one edit of the same size removes between
−2% and +3% of the leak 90% of the time, and never more than 10.4% in 200 tries. Component 1's
18.9% is far outside that range. In `fc6`, random edits swing much more (−12% to +12%, up to
36.5%) because `fc6` sits right before the output, so almost any edit there moves the
prediction. Component 2's 41.4% still beats every random draw, but by a smaller margin.

**Check 2: unit 36.** Killing `fc2` unit 36 removes 27.5% of the leak, but its knob share is
0.94, as high as the deliberate knobs. Row 36 behaves like a volume knob on the model's main
channel. That fits the caution from Stage 01: it pushes the output up, so removing it shrinks a
positive bias, but that says nothing about a trigger.

**Check 3: when each edit acts.** `fc6` component 2 lowers every month by roughly the same
amount (−0.6 to −1.8): a shift, acting the same in January as in July. `fc2` component 1 and
unit 36 both do nothing in winter and act most in the high-flow months, roughly tracking
predicted flow. So `fc2` component 1 is partly knob-like too, as its 0.58 knob share already
said. But it is the least knob-like edit that removes a real share of the leak, and the most
clearly non-random.

### Why are `fc3`–`fc5` inert?

In [13]:
# Why are fc3-fc5 inert? Look at the activations their weights read.
# PCA (plan Appendix C): how many directions does each layer's activity actually vary along?
print(f"{'layer':7}{'PC1':>7}{'PC1-2':>8}{'PC1-3':>8}{'PCs for 99%':>13}")
for name, A in ACTS.items():                       # ACTS from Stage 03; the decoys don't change them
    centred = A - A.mean(axis=0)                   # PCA measures spread about the mean
    s = np.linalg.svd(centred, compute_uv=False)
    explained = s**2 / (s**2).sum()                # share of total variance per direction
    cumulative = np.cumsum(explained)
    n_99 = int(np.searchsorted(cumulative, 0.99)) + 1
    print(f"{name:7}{cumulative[0]:7.3f}{cumulative[1]:8.3f}{cumulative[2]:8.3f}{n_99:13}")

layer      PC1   PC1-2   PC1-3  PCs for 99%
fc1      0.545   0.681   0.802           22
fc2      1.000   1.000   1.000            1
fc3      1.000   1.000   1.000            1
fc4      0.996   1.000   1.000            1
fc5      0.970   0.998   1.000            2
fc6      0.927   0.989   0.999            3


### The network has a bottleneck

After `fc1`, the model's activity varies along essentially **one** direction: a single
principal component holds 100.0% of the variance in `fc2` and `fc3`, 99.6% in `fc4`, 97.0% in
`fc5` and 92.7% in `fc6`. `fc1`, by contrast, needs 22 directions to reach 99%. In plain terms,
`fc1` turns the weather into a rich 22-dimensional description, `fc2` squeezes that into
essentially one number per hour, and the later layers pass that number along.

That explains the scan. The small components of `fc3`–`fc5` read directions their inputs barely
vary along, so they have nothing to react to. It also bears directly on the task. The poison
makes the model react to *"a particular pattern across the hidden units feeding the tampered
layer,"* and the overview warns that separating it from *legitimate learned features* is hard
because of superposition. That description only fits a layer whose inputs carry many features
at once. Only `fc2`'s inputs, the `fc1` activations, do; `fc6`'s inputs carry two.

### Where Stage 04 leaves us

| layer | small components | its inputs vary along (99%) | reading |
| --- | --- | --- | --- |
| `fc2` | component 1 removes 18.9%, far beyond random; partly knob-like (0.58) | **22** directions | **lead suspect** |
| `fc6` | component 2 removes 41.4%, but mostly a shift (0.74); any edit here moves the output | 2 | distant second |
| `fc3`–`fc5` | inert | 1 | ruled out for a feature-triggered poison |

**Caveats, stated plainly.**

- **A poison that reads the main signal itself would be invisible here.** It would sit inside
  component 0, not among the small components, and would act like a flow-dependent knob. The
  overview's description of a feature tangled up with other features argues against this, but
  no measurement here rules it out.
- **`fc2` component 1 is a candidate direction, not the answer.** SVD components are forced to
  be at right angles to each other (Appendix E), so the poison may be spread across several of
  them, and component 1 may capture only part of it.
- **Everything is measured on the public hours.** The scored window is December to May.

### Next

The plan's Stage 05 instruction, *re-do Stage 4 with the decoys removed*, is what this stage just
did. The next real step is Stage 06: train sparse autoencoders on the `fc1` activations (the
18,685 × 56 table feeding `fc2`) and see whether `fc2` component 1's read direction, or something
close to it, keeps turning up.